# KG1 NVIDIA Nemotron - v44 PRODUCTION
## konbu17 LB-validated recipe + NVIDIA best practices + Liger-Kernel fused CE

**Expected score**: 0.68 -> 0.72-0.78 (GPT-5.2/5.3 double-validated, batch=32 regime)

**Based on 5 iterations of research** (double -> quintuple check):
- konbu17 type-weighted sampling (2907 samples) - LB-validated at 291 votes
- lora_alpha=16 + dropout=0.05 (v30/v41 PROVEN for 0.68)
- LR=1e-4 (konbu17 + paper 2602.04998)
- max_length=4096 (konbu17, 4x larger than v43 1024)
- all-linear targeting (v30/v41 PROVEN, PEFT handles out_proj safely)
- Liger-Kernel fused CE (saves 8-12GB memory)
- MoE dtype fix v3 triple-protection (file patch + class patch)
- freeze_moe_router=True (Aman Atar)
- Auto-submit at steps [100, 200, 300, 400] for early feedback


In [ ]:
#@title CELL 0: Cleanup Colab state (v42 leftover removal)
#@markdown Run this FIRST after Runtime restart to ensure clean state.

import os, shutil, subprocess, glob

print("=== v43 Cleanup: removing v42/v40/v39 leftover artifacts ===")

# 1. Kill any lingering training processes
try:
    subprocess.run(["pkill", "-9", "-f", "accelerate"], check=False, timeout=5)
    subprocess.run(["pkill", "-9", "-f", "torch.distributed"], check=False, timeout=5)
    print("  OK Killed lingering training processes")
except Exception as e:
    print("  (skip pkill: " + str(e) + ")")

# 2. Remove stale /tmp directories
stale_dirs = [
    "/tmp/kg1_output",
    "/tmp/kg1_submit",
    "/tmp/kg1_data",
    "/tmp/kg1_repo",
    "/content/kg1-nvidia",
    "/tmp/kg1_strip",
    "/tmp/kg1_ckpt200_new",
    "/tmp/dc4_real_adapter",
    "/tmp/v42_ckpt100",
    "/tmp/v42cells",
]
for d in stale_dirs:
    if os.path.exists(d):
        try:
            shutil.rmtree(d, ignore_errors=True)
            print("  OK Removed " + d)
        except Exception as e:
            print("  (skip " + d + ": " + str(e) + ")")

# 3. Clear HF cache of stale v42/v40/v39 downloads (keep base model cache!)
hf_cache = os.path.expanduser("~/.cache/huggingface/hub")
if os.path.exists(hf_cache):
    patterns = [
        "models--felipesp1983--kg1-nemotron-lora-v4*",
        "models--felipesp1983--kg1-nemotron-lora-v39*",
        "models--felipesp1983--kg1-nemotron-lora-v40*",
    ]
    for pattern in patterns:
        for path in glob.glob(hf_cache + "/" + pattern):
            try:
                shutil.rmtree(path, ignore_errors=True)
                print("  OK Cleared HF cache: " + os.path.basename(path))
            except Exception as e:
                print("  (skip " + path + ": " + str(e) + ")")

# 4. Free GPU memory (if torch already loaded)
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        free_gb = free_bytes / 1e9
        total_gb = total_bytes / 1e9
        print("  OK GPU memory: " + str(round(free_gb, 1)) + "/" + str(round(total_gb, 1)) + " GB free")
except Exception as e:
    print("  (torch not loaded yet: " + str(e) + ")")

# 5. Show disk state
try:
    result = subprocess.run(["df", "-h", "/tmp", "/content"], capture_output=True, text=True, timeout=5)
    print("")
    print("=== Disk state ===")
    print(result.stdout)
except Exception as e:
    print("  (df skip: " + str(e) + ")")

print("")
print("OK CELL 0 CLEANUP COMPLETE - safe to run next cell (Setup)")


In [ ]:
#@title CELL 1: Setup + Config (v44 PRODUCTION)
#@markdown ### v44 = consolidation of konbu17 (LB-validated) + rohanrk1813 (type-specific) + NVIDIA best practices
#@markdown ### Based on 5 iterations of research (double->quintuple check)
#@markdown ### Expected: 0.68 baseline -> 0.74-0.78 target

PHASE = 4  #@param [4] {type:"integer"}
AUTO_SUBMIT = True  #@param {type:"boolean"}
FRESH_LORA = True  #@param {type:"boolean"}

import subprocess, sys, os, json, time, random, zipfile, shutil
from datetime import datetime, timezone
from collections import Counter
from pathlib import Path

# ============================================================
# INSTALL DEPENDENCIES (CRITICAL ORDER)
# ============================================================
print("=== Installing dependencies (v44) ===")

def pip_install(*pkgs):
    for pkg in pkgs:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "--root-user-action=ignore", pkg],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print(f"  OK {pkg}")

# Core stack
pip_install(
    "peft>=0.17.0",
    "datasets",
    "accelerate",
    "trl>=0.17.0",
    "huggingface_hub",
    "kaggle",
    "pandas",
    "vllm>=0.18.0",
)

# Liger-Kernel for fused CE (saves 8-12GB memory) - Agent QC3 validated
print("  Installing liger-kernel (saves 8-12GB memory)...")
try:
    import liger_kernel
    print(f"  OK liger-kernel {liger_kernel.__version__} (cached)")
except ImportError:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "--root-user-action=ignore", "liger-kernel"],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print("  OK liger-kernel installed")
    except Exception as e:
        print(f"  WARN liger-kernel install failed: {e}")
        print("  (training will still work, just without fused CE memory savings)")

# Mamba-ssm (Nemotron hybrid arch requirement)
print("  Installing mamba-ssm (takes ~3min)...")
try:
    import mamba_ssm
    print(f"  OK mamba-ssm {mamba_ssm.__version__} (cached)")
except ImportError:
    r1 = subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "--no-build-isolation", "-q", "--root-user-action=ignore",
         "causal-conv1d==1.5.0.post8", "mamba-ssm==2.2.4"],
        capture_output=True, text=True, timeout=900)
    if r1.returncode != 0:
        print(f"  Strategy 1 FAILED: {r1.stderr[-500:]}")
        r2 = subprocess.run(
            [sys.executable, "-m", "pip", "install",
             "--no-build-isolation", "--root-user-action=ignore",
             "causal-conv1d", "mamba-ssm"],
            capture_output=True, text=True, timeout=1200)
        if r2.returncode != 0:
            print(f"  Strategy 2 FAILED: {r2.stderr[-1000:]}")
            print("  WARN mamba-ssm unavailable - using HF Python fallback")
        else:
            print("  OK Strategy 2 succeeded")
    else:
        print("  OK Strategy 1 succeeded")
    try:
        import mamba_ssm
        print(f"  OK mamba-ssm {mamba_ssm.__version__}")
    except ImportError:
        print("  WARN mamba-ssm not importable - HF will use Python fallback")

import torch
import pandas as pd
from huggingface_hub import HfApi, login, hf_hub_download, snapshot_download

# ============================================================
# MOE DTYPE PRE-PATCH (v44 CRITICAL FIX)
# ============================================================
# From QC3 research: NemotronHTopkRouter returns Float32 weights but
# hidden_states are BFloat16 under quantization. Causes index_add_ crash.
# Fix: patch modeling_nemotron_h.py file BEFORE model load.
print("\n=== Pre-patching MoE dtype bug (v44 triple-protection v3) ===")
try:
    _snap_root = snapshot_download(
        "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16",
        allow_patterns=["*.py", "*.json", "*.txt"],
    )
    print(f"  OK Downloaded Nemotron source files to {_snap_root}")
except Exception as _e:
    print(f"  WARN Download (may already be cached): {_e}")

import glob as _g
_patched_files = 0
for _f in _g.glob("/root/.cache/huggingface/**/modeling_nemotron_h.py", recursive=True):
    try:
        with open(_f, "r") as _fh:
            _c = _fh.read()
        _old = "final_hidden_states.index_add_(0, token_indices, weighted_output)"
        _new = "final_hidden_states.index_add_(0, token_indices, weighted_output.to(final_hidden_states.dtype))"
        if _old in _c and _new not in _c:
            _c = _c.replace(_old, _new)
            with open(_f, "w") as _fh:
                _fh.write(_c)
            _patched_files += 1
            print(f"  OK FILE PATCHED: {_f}")
        elif _new in _c:
            _patched_files += 1
            print(f"  OK FILE already patched: {_f}")
    except Exception as _e:
        print(f"  WARN patch attempt failed for {_f}: {_e}")

if _patched_files == 0:
    print("  WARN No modeling_nemotron_h.py found yet (will rely on CLASS patch in Cell 3)")

# ============================================================
# AUTHENTICATION
# ============================================================
print("\n=== Authentication ===")
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_KEY")
    KAGGLE_USERNAME = userdata.get("KAGGLE_USERNAME")
    KAGGLE_KEY = userdata.get("KAGGLE_KEY")
    print("  OK Colab secrets loaded")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", os.environ.get("HF_KEY", ""))
    KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "felipe1983")
    KAGGLE_KEY = os.environ.get("KAGGLE_KEY", "")
    print("  OK Environment variables loaded")

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("  OK HuggingFace authenticated")

# Kaggle credentials
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("  OK Kaggle credentials configured")

# ============================================================
# GPU INFO
# ============================================================
print("\n=== System ===")
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU: {gpu_name} ({gpu_mem:.0f} GB)")
    if gpu_mem < 40:
        print("  WARN GPU < 40GB. Nemotron 30B needs ~60GB in BF16 or A100 HighRAM/H100.")

# Triton ptxas patch
try:
    ptxas_src = "/usr/local/cuda-12.8/bin/ptxas"
    if not os.path.exists(ptxas_src):
        ptxas_src = "/usr/local/cuda/bin/ptxas"
    if os.path.exists(ptxas_src):
        target = os.path.join(os.path.dirname(shutil.which("python") or "/usr/bin/python"), "ptxas")
        if not os.path.exists(target):
            shutil.copy2(ptxas_src, target)
            print("  OK Triton ptxas patched")
except Exception:
    pass

api = HfApi()

# ============================================================
# v44 PRODUCTION CONFIG (consolidated from 5 iterations)
# ============================================================
MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
DATA_REPO = "felipesp1983/kg1-nemotron-training"

CFG = {
    "name": "v44-production",
    "output_repo": "felipesp1983/kg1-nemotron-lora-v44-production",

    # === DATA (konbu17 LB-validated recipe: 2907 type-weighted samples) ===
    "n_examples": 5000,  # Upper bound; type-weighted sampling decides actual
    "use_konbu17_sampling": True,
    "type_samples": {
        "num": 300,   # Numeral Conversion (20% of 1497)
        "grav": 400,  # Gravitational Constant (27% of 1503)
        "unit": 700,  # Unit Conversion (46% of 1513)
        "enc": 700,   # Text Encryption (47% of 1492)
        "bit": 607,   # Bit Manipulation (100% of 607 after filter)
        "eq": 200,    # Equation Transformation (25% of 787)
    },  # Total: 2907 samples

    # === TRAINING (konbu17 proven on LB) ===
    "n_epochs": 3,            # GPT-5 final: 3 epochs prevents underfitting (+0.02-0.05 expected, +50% time)
    "learning_rate": 7e-5,  # GPT-5 recommendation: sqrt scaling rule for batch=32 (was 1e-4 for batch=8)
    "max_length": 4096,     # konbu17 (4x larger than v43 1024)
    "grad_accum": 32,       # effective batch=32 (GPT-5 recommendation: 30B+MoE needs larger batch)
    "warmup_ratio": 0.08,     # GPT-5 recommendation: larger warmup for larger batch
    "lr_scheduler": "cosine",
    "seed": 123,            # konbu17 exact

    # === TASK FORMAT ===
    "use_thinking": False,  # Match v8/v30 PROVEN (direct boxed answer)
    "use_cot": True,        # Use konbu17 CoT format: "cot\n</think>\n\\boxed{answer}"
    "cot_format": "konbu17",  # "cot + \n</think>\n\\boxed{answer}" wrapping

    # === v44 NEW OPTIMIZATIONS (from QC3) ===
    "use_liger_kernel": True,      # Fused CE saves 8-12GB memory
    "activation_offloading": True, # Saves 3-5GB via CPU offload

    # === MoE NONUPLE FLAGS (v43/v42 proven) ===
    "freeze_moe_router": True,      # Aman Atar 168 votes; prevents router collapse
    "exclude_out_proj": False,      # Use "all-linear" (v30/v41 PROVEN baseline)
    "checkpoint_averaging": True,   # AIMO-2 free +0.005-0.020

    # === GATES (feedback_99percent_rule.md) ===
    "skip_pretrain_smoke": False,
    "skip_prescore_gate": False,
    "prescore_min_threshold": 0.60,

    # === SUBMIT STRATEGY ===
    "submit_steps": [100, 200, 300, 400],  # 4 submits across training (step 100 early)

    # === WARM START (konbu17 is fresh) ===
    "warm_start_repo": None,
}

OUTPUT_REPO = CFG["output_repo"]
OUTPUT_DIR = f"/tmp/kg1_output/{CFG['name']}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === FIXED LORA CONFIG ===
LORA_RANK = 32           # Hard limit by Kaggle competition (max_lora_rank=32)
LORA_ALPHA = 16          # v30/v41 PROVEN ratio 0.5 (v30 = 0.68 LB)
LORA_DROPOUT = 0.05      # konbu17 + v30 consensus

# ============================================================
# CONFIG FLAGS (Cell 3 / 3.5 dependencies)
# ============================================================
FREEZE_MOE_ROUTER = CFG["freeze_moe_router"]
EXCLUDE_OUT_PROJ = CFG["exclude_out_proj"]
CHECKPOINT_AVERAGING = CFG["checkpoint_averaging"]
SKIP_PRETRAIN_SMOKE = CFG["skip_pretrain_smoke"]
SKIP_PRESCORE_GATE = CFG["skip_prescore_gate"]
PRESCORE_MIN = CFG["prescore_min_threshold"]
USE_LIGER_KERNEL = CFG["use_liger_kernel"]

# ============================================================
# SUMMARY
# ============================================================
print("\n" + "=" * 60)
print(f"  KG1 v44 PRODUCTION - {CFG['name']}")
print("=" * 60)
print(f"  Model:         {MODEL_NAME}")
print(f"  Examples:      {sum(CFG['type_samples'].values())} (konbu17 type-weighted)")
print(f"  Epochs:        {CFG['n_epochs']}")
print(f"  LR:            {CFG['learning_rate']} (konbu17 LB-validated)")
print(f"  MaxLen:        {CFG['max_length']} (4x v43)")
print(f"  LoRA:          r={LORA_RANK} a={LORA_ALPHA} dropout={LORA_DROPOUT}")
print(f"  Liger Kernel:  {USE_LIGER_KERNEL} (saves 8-12GB)")
print(f"  Freeze router: {FREEZE_MOE_ROUTER}")
print(f"  Ckpt average:  {CHECKPOINT_AVERAGING}")
print(f"  Submit steps:  {CFG['submit_steps']}")
print(f"  Warm start:    {CFG['warm_start_repo'] or 'Fresh'}")
print(f"  Output repo:   {OUTPUT_REPO}")
print("=" * 60)

print("\nOK CELL 1 COMPLETE")


In [ ]:
#@title CELL 2: Load & Prepare Data (v44 konbu17 recipe)

# ============================================================
# DOWNLOAD TRAINING DATA
# ============================================================
print("=== Downloading training data ===")

hf_hub_download(repo_id=DATA_REPO, filename="data/train.csv", local_dir="/tmp/kg1_data")
train_df = pd.read_csv("/tmp/kg1_data/data/train.csv")
print(f"  OK Official data: {len(train_df)} rows")

# Load wrong_ids (post-sampling filter, v41 pattern)
wrong_ids = set()
try:
    hf_hub_download(repo_id=DATA_REPO, filename="train_verified.csv",
                    local_dir="/tmp/kg1_data", repo_type="dataset")
    verified_df = pd.read_csv("/tmp/kg1_data/train_verified.csv")
    wrong_ids = set(verified_df[verified_df["status"] == "verified_wrong"]["id"])
    print(f"  OK wrong_ids loaded: {len(wrong_ids)}")
except Exception as e:
    print(f"  WARN train_verified.csv not found ({e})")

# ============================================================
# CLASSIFY FAMILIES (same as v30)
# ============================================================
def classify(prompt):
    p = prompt.lower()
    if "bit manipulation" in p: return "bit"
    if "gravitational" in p or "gravity" in p: return "grav"
    if "unit conversion" in p or "measurement" in p: return "unit"
    if "numeral" in p: return "num"
    if "encryption" in p or "cipher" in p: return "enc"
    if "transformation" in p: return "eq"
    return "other"

train_df["family"] = train_df["prompt"].apply(classify)
train_df["ans_len"] = train_df["answer"].astype(str).str.len()

# Filter answer length (Kaggle max_answer_length=24 from competition_utils.py DATA_GATE_POLICY)
filtered_df = train_df[train_df["ans_len"] <= 24].copy()
print(f"  After ans_len <= 24 filter: {len(filtered_df)}")

print("\n  Family distribution in pool:")
print(filtered_df["family"].value_counts().to_string())

# ============================================================
# v44 KONBU17 TYPE-WEIGHTED SAMPLING
# ============================================================
# konbu17 LB-validated recipe: Bit=607 (all), Eq=200, Enc=700, Unit=700, Grav=400, Num=300
# Total = 2907 balanced samples
random.seed(CFG["seed"])

type_samples = CFG["type_samples"]
print(f"\n=== v44 Type-weighted sampling (konbu17 LB-validated) ===")

sampled_dfs = []
for fam, n_target in type_samples.items():
    fam_df = filtered_df[filtered_df["family"] == fam]
    avail = len(fam_df)
    actual = min(n_target, avail)
    if actual > 0:
        if actual == avail:
            sampled = fam_df  # Take all
        else:
            sampled = fam_df.sample(n=actual, random_state=CFG["seed"])
        sampled_dfs.append(sampled)
        print(f"  {fam:8s}: target={n_target:3d}, available={avail:4d}, actual={actual:3d}")
    else:
        print(f"  {fam:8s}: NOT FOUND in data")

train_sampled = pd.concat(sampled_dfs, ignore_index=True) if sampled_dfs else pd.DataFrame()
print(f"\n  Total sampled: {len(train_sampled)}")

# Shuffle (konbu17 seed=123)
train_sampled = train_sampled.sample(frac=1, random_state=CFG["seed"]).reset_index(drop=True)

# v41 fix: apply verified_wrong filter AFTER sampling (prevents seed shift)
if wrong_ids:
    before_filter = len(train_sampled)
    train_sampled = train_sampled[~train_sampled["id"].isin(wrong_ids)].reset_index(drop=True)
    dropped = before_filter - len(train_sampled)
    if dropped > 0:
        print(f"  Dropped {dropped} verified_wrong (subset of konbu17 sampling)")

# ============================================================
# BUILD SFT RECORDS (konbu17 format)
# ============================================================
# PROMPT_SUFFIX EXACTLY matches OFFICIAL_PROMPT_SUFFIX from src/competition_utils.py
PROMPT_SUFFIX = "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"

import re as _re

examples = []
skipped_no_cot = 0

# Check if 'generated_cot' column exists (konbu17 dataset has this)
has_cot_column = "generated_cot" in train_sampled.columns or "cot" in train_sampled.columns
cot_column = "generated_cot" if "generated_cot" in train_sampled.columns else ("cot" if "cot" in train_sampled.columns else None)

if has_cot_column:
    print(f"\n  OK Found CoT column: {cot_column} (using konbu17 CoT format)")
else:
    print("\n  WARN No CoT column in data - using direct boxed format (v8/v30 fallback)")

for _, row in train_sampled.iterrows():
    prompt = str(row["prompt"])
    answer = str(row["answer"])
    user_content = prompt + PROMPT_SUFFIX

    if has_cot_column and CFG["use_cot"]:
        cot = str(row[cot_column])
        if not cot or cot == "nan" or len(cot.strip()) < 5:
            skipped_no_cot += 1
            continue
        # Strip any existing \boxed{} from CoT (konbu17 pattern)
        cot_cleaned = _re.sub(r"\\boxed\{[^}]*\}", "", cot).rstrip()
        # v44 GPT-5 clean format: <think>cot</think>oxed{answer} (Nemotron pretrain aligned)
        assistant_content = f"<think>\n{cot_cleaned}\n</think>\n\boxed{{{answer}}}"
    else:
        # Direct format (v8/v30 fallback): just \boxed{answer}
        assistant_content = f"\\boxed{{{answer}}}"

    examples.append({
        "messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ]
    })

print(f"\n  Built {len(examples)} examples (skipped {skipped_no_cot} without CoT)")

# ============================================================
# FINAL STATS
# ============================================================
def _classify_example(ex):
    for m in ex["messages"]:
        if m.get("role") == "user":
            return classify(m["content"])
    return "other"

fam_counts = Counter(_classify_example(e) for e in examples)
print(f"\n=== v44 Final Dataset: {len(examples)} examples ===")
for fam, cnt in sorted(fam_counts.items()):
    pct = cnt / len(examples) * 100 if examples else 0
    print(f"  {fam:8s}: {cnt:4d} ({pct:.1f}%)")

# Verify format
print("\n=== Sample Example (first) ===")
if examples:
    _sample = examples[0]["messages"]
    _user = next((m for m in _sample if m.get("role") == "user"), _sample[0])
    _asst = next((m for m in _sample if m.get("role") == "assistant"), _sample[-1])
    print(f"USER (first 200 chars): {_user['content'][:200]}...")
    print(f"ASSISTANT (first 200 chars): {_asst['content'][:200]}")

print("\nOK CELL 2 COMPLETE")


In [ ]:
#@title CELL 3: Load Model + LoRA (v44 with MoE triple-protection)

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import Dataset

# ============================================================
# LOAD BASE MODEL (BF16)
# ============================================================
print("=== Loading Nemotron-3-Nano-30B (BF16) ===")
print("  (This takes ~3 minutes on H100, ~5 min on A100 HighRAM)")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map={"": 0},      # Force GPU 0 (no meta tensors)
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
print(f"  OK Model loaded: {model.num_parameters()/1e9:.1f}B params")

# Disable Mamba fast path (known to cause issues in training - v30/v41 proven)
fast_path_count = 0
for module in model.modules():
    if hasattr(module, "is_fast_path_available"):
        module.is_fast_path_available = False
        fast_path_count += 1
print(f"  OK Fast path disabled ({fast_path_count} modules)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"  OK Tokenizer loaded (vocab={tokenizer.vocab_size})")

# ============================================================
# MOE DTYPE FIX v3 - CLASS-LEVEL PATCH (v44 triple-protection)
# ============================================================
# Fix for: NemotronHTopkRouter returns Float32 topk_weights while
# hidden_states is BFloat16 under NF4 + autocast. Pre-patch in Cell 1
# handled file-level; this is class-level backup.
print("\n=== Applying MoE dtype fix v3 (class-level patch) ===")
_moe_class_patched = set()
for _name, _module in model.named_modules():
    _cls = type(_module)
    _cls_name = _cls.__name__
    if _cls_name in _moe_class_patched:
        continue
    if not hasattr(_cls, "moe") or not callable(getattr(_cls, "moe", None)):
        continue

    def _make_class_patch():
        def _patched_moe(self, hidden_states, topk_indices, topk_weights):
            import torch as _t
            final_hidden_states = _t.zeros_like(hidden_states, dtype=hidden_states.dtype)
            expert_mask = _t.nn.functional.one_hot(
                topk_indices, num_classes=len(self.experts)
            )
            expert_mask = expert_mask.permute(2, 0, 1)
            for expert_idx in range(len(self.experts)):
                expert = self.experts[expert_idx]
                mask = expert_mask[expert_idx]
                token_indices, weight_indices = _t.where(mask)
                if token_indices.numel() > 0:
                    expert_weights = topk_weights[token_indices, weight_indices]
                    expert_input = hidden_states[token_indices]
                    expert_output = expert(expert_input)
                    weighted_output = expert_output * expert_weights.unsqueeze(-1)
                    # CRITICAL: cast to hidden_states.dtype before index_add
                    final_hidden_states.index_add_(
                        0, token_indices,
                        weighted_output.to(final_hidden_states.dtype),
                    )
            return final_hidden_states.type(hidden_states.dtype)
        return _patched_moe

    _cls.moe = _make_class_patch()
    _moe_class_patched.add(_cls_name)
    print(f"  OK MoE dtype fix CLASS PATCHED: {_cls_name}")

if not _moe_class_patched:
    print("  INFO No MoE class found to patch (file patch may have been sufficient)")

# ============================================================
# FREEZE MOE ROUTERS (NONUPLE - Aman Atar 168 votes)
# ============================================================
if FREEZE_MOE_ROUTER:
    frozen_router_count = 0
    for name, param in model.named_parameters():
        # Match "router" or "gate" but NOT "gate_proj" (that's MLP, we want that LoRA'd)
        if "router" in name.lower() or ("gate" in name.lower() and "gate_proj" not in name.lower()):
            param.requires_grad = False
            frozen_router_count += 1
    print(f"  OK NONUPLE: Frozen {frozen_router_count} MoE router params")
else:
    print("  (skipping freeze_moe_router)")

# ============================================================
# APPLY LORA (v30/v41 PROVEN all-linear baseline)
# ============================================================
# v44 DECISION (from QC2+QC3 research):
# - "all-linear" is v30/v41 PROVEN (0.68 LB)
# - konbu17 regex (in_proj|out_proj|up_proj|down_proj) EXCLUDES attention
#   which has only 6 layers but provides 0.42% of params
# - PEFT #2274: out_proj Mamba2 is BROKEN under LoRA (kernel bypass)
#   but "all-linear" doesn't crash, PEFT just silently skips incompatible
# - Using "all-linear" is SAFEST: covers attention q/k/v/o + Mamba in_proj
#   + FFN up/down/gate. out_proj LoRA weights won't apply but won't crash
print(f"\n=== Applying LoRA (r={LORA_RANK}, α={LORA_ALPHA}) ===")

adapter_loaded = False
if not adapter_loaded:
    print("  Creating fresh LoRA (all-linear, v30/v41 PROVEN baseline)...")
    model.enable_input_require_grads()

    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules="all-linear",  # v30/v41 PROVEN (0.68 LB)
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    print("  OK Fresh LoRA applied (all-linear targeting)")

model.print_trainable_parameters()

# ============================================================
# PREPARE TOKENIZED DATASET
# ============================================================
print("\n=== Preparing dataset for SFTTrainer ===")

# Use apply_chat_template (tokenizer handles Nemotron's template automatically)
# IMPORTANT: v44 uses add_generation_prompt=False during training (matches v8)
# Inference will use enable_thinking=True but that's handled by vLLM chat_template
texts = []
for ex in examples:
    text = tokenizer.apply_chat_template(
        ex["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    texts.append(text)

ds = Dataset.from_dict({"text": texts})
print(f"  Dataset: {len(ds)} examples")

# Token length stats (sample 100)
sample_lens = [len(tokenizer(t)["input_ids"]) for t in texts[:100]]
print(f"  Token lengths (sample 100): min={min(sample_lens)}, max={max(sample_lens)}, mean={sum(sample_lens)/len(sample_lens):.0f}")

# Over-length warning
over_len = sum(1 for t in texts if len(tokenizer(t)["input_ids"]) > CFG["max_length"])
if over_len > 0:
    print(f"  WARN {over_len} examples exceed max_length={CFG['max_length']} (will be truncated)")

# Verify template format
print("\n=== Template Sample (first 500 chars) ===")
print(texts[0][:500])

print("\nOK CELL 3 COMPLETE")


In [ ]:
#@title 🧪 CELL 3.5: Pre-treino Smoke Test + Pre-score Gate (NONUPLE)
#@markdown ### Mandatory smoke test from feedback memory: 2 steps + pre-score before paid job
#@markdown Set SKIP_SMOKE=True to bypass for v30/v41 backward compat

if SKIP_PRETRAIN_SMOKE and SKIP_PRESCORE_GATE:
    print("⏭️  CELL 3.5 SKIPPED (Phase 1-3 backward compat)")
    smoke_test_passed = True
    prescore_passed = True
else:
    from transformers import TrainerCallback
    from trl import SFTTrainer, SFTConfig
    import time, gc, torch

    print("=" * 60)
    print("  NONUPLE PRE-TREINO SMOKE TEST + PRE-SCORE GATE")
    print("  (mandatory by feedback_pretreino_prescore.md)")
    print("=" * 60)

    # =============== PRE-TREINO SMOKE TEST: 2 STEPS ===============
    if not SKIP_PRETRAIN_SMOKE:
        print("\n[1/2] PRE-TREINO SMOKE TEST (2 steps)...")

        # Use small subset for smoke test (10 examples = 2 steps with grad_accum=8)
        smoke_ds = ds.select(range(min(20, len(ds))))
        smoke_args = SFTConfig(
            output_dir="/tmp/smoke_test",
            dataset_text_field="text",
            max_length=CFG["max_length"],
            packing=False,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=CFG["grad_accum"],
            max_steps=2,                              # ONLY 2 STEPS
            learning_rate=CFG["learning_rate"],
            warmup_steps=0,
            bf16=True,
            logging_steps=1,
            save_strategy="no",                       # don't save smoke checkpoint
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            report_to="none",
            dataloader_num_workers=0,
            max_grad_norm=1.0,
        )

        smoke_trainer = SFTTrainer(
            model=model,
            train_dataset=smoke_ds,
            processing_class=tokenizer,
            args=smoke_args,
        )

        smoke_start = time.time()
        try:
            smoke_trainer.train()
            smoke_elapsed = time.time() - smoke_start

            # Verify loss is reasonable
            if smoke_trainer.state.log_history:
                last_loss = smoke_trainer.state.log_history[-1].get("loss", 999)
                if last_loss > 8.0:
                    print(f"  ❌ SMOKE FAIL: loss {last_loss:.2f} > 8.0 (something wrong)")
                    smoke_test_passed = False
                else:
                    print(f"  ✅ SMOKE PASS: 2 steps in {smoke_elapsed:.0f}s, loss={last_loss:.2f}")
                    smoke_test_passed = True
            else:
                print(f"  ⚠️  SMOKE: no loss logged (uncertain)")
                smoke_test_passed = True  # be lenient
        except Exception as e:
            print(f"  ❌ SMOKE FAIL: {e}")
            smoke_test_passed = False

        # CRITICAL: clean up smoke trainer to free memory before real training
        del smoke_trainer, smoke_args, smoke_ds
        gc.collect()
        torch.cuda.empty_cache()
    else:
        smoke_test_passed = True
        print("\n[1/2] Smoke test SKIPPED")

    # =============== PRE-SCORE GATE (lightweight) ===============
    # Note: full pre-score requires vLLM + adapter eval which takes ~10min.
    # For v42 we only check that the loss is sensible (smoke test = de facto pre-score).
    # Full pre-score happens via scripts/prescore_submission_candidate.py AFTER training.
    if not SKIP_PRESCORE_GATE:
        print(f"\n[2/2] PRE-SCORE GATE (using smoke test as proxy)...")
        # Full pre-score requires vLLM + adapter eval (~10min, not worth before paid run)
        # Smoke test loss < 8.0 is used as lightweight proxy.
        prescore_passed = smoke_test_passed
        print(f"  ✓ PRE-SCORE GATE: smoke_test_passed={smoke_test_passed} -> prescore_passed={prescore_passed}")
    else:
        prescore_passed = True
        print("\n[2/2] Pre-score gate SKIPPED")

    # =============== FINAL DECISION ===============
    if not smoke_test_passed or not prescore_passed:
        print("\n" + "=" * 60)
        print("  ❌ NONUPLE GATE FAILED — STOPPING BEFORE PAID TRAINING")
        print(f"  smoke_test_passed: {smoke_test_passed}")
        print(f"  prescore_passed: {prescore_passed}")
        print("  Investigate the issue before retrying.")
        print("=" * 60)
        raise RuntimeError("NONUPLE pre-training gate failed - investigate before paid run")
    else:
        print("\n" + "=" * 60)
        print("  ✅ NONUPLE PRE-TREINO + PRE-SCORE GATES PASSED")
        print("  Safe to proceed to full training (Cell 4)")
        print("=" * 60)

print("\n✅ CELL 3.5 COMPLETE")


In [ ]:
#@title CELL 4: Train (SFT) + Auto-Submit (v44 with Liger-Kernel)

from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
from safetensors.torch import load_file, save_file

# ============================================================
# KAGGLE SUBMISSION FUNCTIONS
# ============================================================
COMPETITION = "nvidia-nemotron-model-reasoning-challenge"

def strip_moe_experts(adapter_dir, stripped_dir):
    """Remove MoE expert LoRA weights, keep attn+mamba+shared+gate.
    Matches v30 strip pattern (proven 0.68 LB)."""
    os.makedirs(stripped_dir, exist_ok=True)
    adapter_path = os.path.join(adapter_dir, "adapter_model.safetensors")
    config_path = os.path.join(adapter_dir, "adapter_config.json")

    if not os.path.exists(adapter_path):
        raise FileNotFoundError(f"adapter_model.safetensors not found in {adapter_dir}")

    tensors = load_file(adapter_path)
    n_total = len(tensors)

    keep = {k: v for k, v in tensors.items() if "expert" not in k.lower()}
    n_keep = len(keep)
    n_strip = n_total - n_keep
    keep_size_mb = sum(v.numel() * v.element_size() for v in keep.values()) / 1e6
    print(f"    Strip: {n_total}->{n_keep} keys ({n_strip} stripped, {keep_size_mb:.1f} MB)")

    out_safetensors = os.path.join(stripped_dir, "adapter_model.safetensors")
    save_file(keep, out_safetensors)

    with open(config_path) as f:
        config = json.load(f)

    keep_modules = set()
    for key in keep:
        for p in key.split("."):
            if p in ("q_proj", "k_proj", "v_proj", "o_proj",
                     "in_proj", "out_proj", "up_proj", "down_proj", "gate_proj"):
                keep_modules.add(p)
    config["target_modules"] = sorted(keep_modules)

    out_config = os.path.join(stripped_dir, "adapter_config.json")
    with open(out_config, "w") as f:
        json.dump(config, f, indent=2)

    return stripped_dir


def create_submission_zip(adapter_dir, zip_path):
    """Strip MoE experts then create submission.zip."""
    os.makedirs(os.path.dirname(zip_path), exist_ok=True)
    base_name = os.path.basename(zip_path).replace(".zip", "")
    stripped_dir = os.path.join(os.path.dirname(zip_path), f"stripped_{base_name}")
    strip_moe_experts(adapter_dir, stripped_dir)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fname in ["adapter_config.json", "adapter_model.safetensors"]:
            fpath = os.path.join(stripped_dir, fname)
            if os.path.exists(fpath):
                size_mb = os.path.getsize(fpath) / 1e6
                zf.write(fpath, fname)
                print(f"    Added {fname} ({size_mb:.1f} MB)")
    size_mb = os.path.getsize(zip_path) / 1e6
    print(f"    Zip: {zip_path} ({size_mb:.1f} MB)")
    return zip_path


def submit_to_kaggle(zip_path, description):
    """Submit adapter zip to Kaggle."""
    try:
        result = subprocess.run(
            ["kaggle", "competitions", "submit",
             "-c", COMPETITION,
             "-f", zip_path,
             "-m", description],
            capture_output=True, text=True, timeout=600
        )
        if result.returncode == 0:
            print(f"    OK SUBMITTED: {description}")
            return True
        else:
            print(f"    FAIL SUBMIT: {result.stderr[:200]}")
            return False
    except Exception as e:
        print(f"    FAIL SUBMIT ERROR: {e}")
        return False

# ============================================================
# TRAINING CALLBACKS
# ============================================================
class CheckpointSubmitCallback(TrainerCallback):
    """Upload to HF + submit to Kaggle at specific steps."""
    def __init__(self, repo_id, submit_steps, auto_submit=True):
        self.repo_id = repo_id
        self.submit_steps = set(submit_steps)
        self.auto_submit = auto_submit
        self.api = HfApi()
        self.submitted = set()
        try:
            self.api.create_repo(repo_id, private=True, exist_ok=True)
        except Exception:
            pass

    def on_save(self, args, state, control, **kwargs):
        import glob as g
        step = state.global_step
        loss = state.log_history[-1].get("loss", "N/A") if state.log_history else "N/A"

        ckpts = sorted(g.glob(f"{args.output_dir}/checkpoint-*"))
        if not ckpts:
            return
        ckpt_dir = ckpts[-1]

        try:
            self.api.upload_folder(
                folder_path=ckpt_dir,
                path_in_repo=f"checkpoint-{step}",
                repo_id=self.repo_id,
                commit_message=f"Step {step} | Loss {loss} | Epoch {state.epoch:.2f}",
            )
            print(f"\n>>> HF Upload: step {step}, loss={loss}")
        except Exception as e:
            print(f"\n>>> HF Upload failed: {e}")

        if self.auto_submit and step in self.submit_steps and step not in self.submitted:
            print(f"\n{'='*50}")
            print(f"=== KAGGLE SUBMIT (step {step}) ===")
            print(f"{'='*50}")
            zip_path = f"/tmp/kg1_submit/submission_step{step}.zip"
            try:
                create_submission_zip(ckpt_dir, zip_path)
                desc = f"{CFG['name']} step-{step} loss-{loss} r{LORA_RANK}-a{LORA_ALPHA} konbu17-recipe"
                if submit_to_kaggle(zip_path, desc):
                    self.submitted.add(step)
            except Exception as e:
                print(f"    FAIL Submit pipeline error: {e}")

    def on_log(self, args, state, control, logs=None, **kwargs):
        """Early warning: loss at step 10 > 20.0 is bad (v44 TRL17 calibration)."""
        if state.global_step == 10 and logs:
            loss = logs.get("loss", 0)
            if loss > 20.0:
                print(f"\nWARN ALERT: Loss step 10 = {loss:.2f} (muito alta!)")
                print("v44 threshold: v8 had loss ~3-4 at step 10.")
                print("Continuing but monitor closely...")
            elif loss > 15.0:
                print(f"\nWARN Loss step 10 = {loss:.2f} (elevated, v8 had ~3-4)")
            else:
                print(f"\nOK Loss step 10 = {loss:.2f} (compatible with v8 baseline)")

# ============================================================
# TRAINING CONFIG (v44 konbu17 recipe + Liger-Kernel)
# ============================================================
# Build kwargs conditionally (liger may not be installed)
sft_kwargs = dict(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=CFG["max_length"],
    packing=False,
    num_train_epochs=CFG["n_epochs"],
    per_device_train_batch_size=1,
    gradient_accumulation_steps=CFG["grad_accum"],
    learning_rate=CFG["learning_rate"],
    warmup_ratio=CFG["warmup_ratio"],
    weight_decay=0.01,
    lr_scheduler_type=CFG["lr_scheduler"],
    optim="adamw_torch",
    bf16=True,
    logging_steps=5,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=5,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    dataloader_num_workers=0,
    max_grad_norm=1.0,
    seed=CFG["seed"],
    remove_unused_columns=False,
)

# Try to enable Liger-Kernel (fused CE saves 8-12GB memory)
if USE_LIGER_KERNEL:
    try:
        import liger_kernel  # noqa: F401
        sft_kwargs["use_liger_kernel"] = True
        print("  OK Liger-Kernel enabled (fused CE saves 8-12GB)")
    except ImportError:
        print("  WARN liger-kernel not installed, skipping fused CE optimization")

training_args = SFTConfig(**sft_kwargs)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    processing_class=tokenizer,
    args=training_args,
    callbacks=[CheckpointSubmitCallback(
        OUTPUT_REPO,
        CFG["submit_steps"],
        auto_submit=AUTO_SUBMIT
    )],
)

# ============================================================
# TRAINING ESTIMATES
# ============================================================
total_steps = max(1, len(ds) // CFG["grad_accum"] * CFG["n_epochs"])
est_h = total_steps * 45 / 3600  # ~45s/step on H100
print(f"\n{'='*60}")
print(f"  TRAINING: {CFG['name']}")
print(f"  Examples: {len(ds)} | Epochs: {CFG['n_epochs']}")
print(f"  Steps: ~{total_steps} | Est. time: ~{est_h:.1f}h")
print(f"  Batch size: 1 x {CFG['grad_accum']} grad_accum = {CFG['grad_accum']} effective")
print(f"  Max length: {CFG['max_length']} tokens")
print(f"  Auto-submit at steps: {sorted(CFG['submit_steps'])}")
print(f"  Warm start: {'YES' if adapter_loaded else 'NO (fresh LoRA)'}")
print(f"{'='*60}")

# ============================================================
# TRAIN!
# ============================================================
print("\nStarting training...")
start_time = time.time()

try:
    trainer.train()
    print("\nOK Training complete!")
except Exception as e:
    print(f"\nFAIL Training error: {e}")
    # Emergency save
    try:
        emergency_dir = f"{OUTPUT_DIR}/emergency"
        model.save_pretrained(emergency_dir)
        tokenizer.save_pretrained(emergency_dir)
        api.upload_folder(folder_path=emergency_dir, repo_id=OUTPUT_REPO,
                         path_in_repo="emergency",
                         commit_message=f"Emergency save: {str(e)[:80]}")
        print("  Emergency save uploaded to HF")
    except Exception:
        print("  Emergency save failed")

elapsed = time.time() - start_time
print(f"\nTime: {elapsed/3600:.2f}h")

# ============================================================
# SAVE & UPLOAD FINAL
# ============================================================
print("\n=== Saving final adapter ===")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

final_loss = "N/A"
if trainer.state.log_history:
    for entry in reversed(trainer.state.log_history):
        if "loss" in entry:
            final_loss = entry["loss"]
            break

status = {
    "version": CFG["name"],
    "phase": PHASE,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "examples": len(examples),
    "epochs": CFG["n_epochs"],
    "lr": CFG["learning_rate"],
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "max_length": CFG["max_length"],
    "grad_accum": CFG["grad_accum"],
    "recipe": "konbu17-v44-production",
    "training_time_h": elapsed / 3600,
    "final_loss": final_loss,
    "total_steps": trainer.state.global_step,
}
with open(f"{OUTPUT_DIR}/adapter_status.json", "w") as f:
    json.dump(status, f, indent=2)

# Upload final
print("\n=== Uploading final to HF ===")
try:
    api.create_repo(OUTPUT_REPO, private=True, exist_ok=True)
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=OUTPUT_REPO,
        commit_message=f"FINAL v44: {len(examples)}ex, {CFG['n_epochs']}ep, loss={final_loss}, {elapsed/3600:.1f}h",
    )
    print(f"  OK Uploaded to: https://huggingface.co/{OUTPUT_REPO}")
except Exception as e:
    print(f"  FAIL Upload failed: {e}")

# Final submission
if AUTO_SUBMIT:
    print("\n=== FINAL KAGGLE SUBMISSION ===")
    zip_path = f"/tmp/kg1_submit/submission_final.zip"
    try:
        create_submission_zip(OUTPUT_DIR, zip_path)
        desc = f"{CFG['name']} FINAL loss-{final_loss} r{LORA_RANK}-a{LORA_ALPHA} konbu17 {len(examples)}ex {CFG['n_epochs']}ep"
        submit_to_kaggle(zip_path, desc)
    except Exception as e:
        print(f"  FAIL Final submit error: {e}")

print(f"\n{'='*60}")
print(f"  v44 TRAINING SUMMARY")
print(f"  Phase: {PHASE} ({CFG['name']})")
print(f"  Final loss: {final_loss}")
print(f"  Total steps: {trainer.state.global_step}")
print(f"  Time: {elapsed/3600:.2f}h")
print(f"  Output: {OUTPUT_REPO}")
print(f"{'='*60}")
print("\nOK CELL 4 COMPLETE")


In [ ]:
#@title 🔀 CELL 4.5: Checkpoint Averaging + Per-Family Eval (NONUPLE)
#@markdown ### AIMO-2 trick: linearly merge last 4 checkpoints for +1-2 score points free

if not CHECKPOINT_AVERAGING:
    print("⏭️  CELL 4.5 SKIPPED (Phase 1-3 backward compat)")
else:
    import glob, torch
    from safetensors.torch import load_file, save_file
    from huggingface_hub import HfApi

    print("=" * 60)
    print("  NONUPLE CHECKPOINT AVERAGING (AIMO-2 trick)")
    print("=" * 60)

    # Find all checkpoints in OUTPUT_DIR
    ckpts = sorted(
        glob.glob(f"{OUTPUT_DIR}/checkpoint-*"),
        key=lambda p: int(p.rsplit("-", 1)[-1]),
    )
    print(f"  Found {len(ckpts)} checkpoints in {OUTPUT_DIR}")

    if len(ckpts) < 2:
        print(f"  ⚠️  Need >= 2 checkpoints for averaging, found {len(ckpts)}")
        print(f"  ⏭️  Skipping averaging")
    else:
        # Use last 4 (or all if fewer)
        ckpts_to_average = ckpts[-4:]
        print(f"  Averaging {len(ckpts_to_average)} checkpoints:")
        for c in ckpts_to_average:
            print(f"    {c}")

        # Load all adapter_model.safetensors
        states = []
        for c in ckpts_to_average:
            adapter_file = f"{c}/adapter_model.safetensors"
            if not os.path.exists(adapter_file):
                print(f"  ⚠️  Skipping {c}: no adapter_model.safetensors")
                continue
            states.append(load_file(adapter_file))

        if len(states) < 2:
            print(f"  ❌ Could not load enough valid checkpoints")
        else:
            # Linear average
            avg_state = {}
            for key in states[0].keys():
                avg_state[key] = sum(s[key].float() for s in states) / len(states)
                # Restore original dtype (bfloat16)
                avg_state[key] = avg_state[key].to(states[0][key].dtype)

            # Save averaged adapter
            avg_dir = f"{OUTPUT_DIR}/averaged"
            os.makedirs(avg_dir, exist_ok=True)
            save_file(avg_state, f"{avg_dir}/adapter_model.safetensors")

            # Copy adapter_config.json from latest checkpoint
            import shutil
            shutil.copy2(f"{ckpts_to_average[-1]}/adapter_config.json", f"{avg_dir}/adapter_config.json")

            print(f"  ✓ Averaged adapter saved to {avg_dir}")

            # Upload averaged to HF
            try:
                api = HfApi()
                api.create_repo(OUTPUT_REPO, private=True, exist_ok=True)
                api.upload_folder(
                    folder_path=avg_dir,
                    path_in_repo="averaged",
                    repo_id=OUTPUT_REPO,
                    commit_message=f"AIMO-2 averaged: {len(states)} checkpoints",
                )
                print(f"  ✓ Uploaded averaged to https://huggingface.co/{OUTPUT_REPO}/tree/main/averaged")
            except Exception as e:
                print(f"  ⚠️  Upload failed: {e}")

            # Auto-submit averaged version
            if AUTO_SUBMIT:
                print(f"\n  === Submitting AVERAGED to Kaggle ===")
                avg_zip = f"/tmp/kg1_submit/submission_averaged.zip"
                try:
                    create_submission_zip(avg_dir, avg_zip)
                    desc = f"{CFG['name']} AVERAGED ({len(states)} ckpts) NONUPLE"
                    submit_to_kaggle(avg_zip, desc)
                except Exception as e:
                    print(f"  ❌ Averaged submit error: {e}")

    print("\n✅ CELL 4.5 COMPLETE")


In [ ]:
#@title 📈 CELL 5: Check Scores + Diagnostics

print("=== Checking Kaggle submissions ===")
try:
    result = subprocess.run(
        ["kaggle", "competitions", "submissions",
         "-c", COMPETITION, "--csv"],
        capture_output=True, text=True, timeout=30
    )
    if result.returncode == 0:
        lines = result.stdout.strip().split("\n")
        print(f"\nRecent submissions:")
        for line in lines[:10]:
            print(f"  {line}")
    else:
        print(f"Error: {result.stderr}")
except Exception as e:
    print(f"Error checking submissions: {e}")

# Training loss curve
if trainer.state.log_history:
    print(f"\n=== Training Loss Curve ===")
    losses = [(h.get("step", 0), h.get("loss", None))
              for h in trainer.state.log_history if h.get("loss") is not None]
    for step, loss in losses:
        bar = "█" * int(max(0, 40 - loss * 5))
        print(f"  Step {step:5d}: loss={loss:.4f} {bar}")
    
    if losses:
        first_loss = losses[0][1]
        last_loss = losses[-1][1]
        print(f"\n  First loss: {first_loss:.4f}")
        print(f"  Final loss: {last_loss:.4f}")
        print(f"  Reduction: {(1-last_loss/first_loss)*100:.1f}%")
        
        if last_loss < 1.0:
            print(f"  ✅ Excelente convergência! Score esperado: 0.70+")
        elif last_loss < 2.0:
            print(f"  ✅ Boa convergência. Score esperado: 0.65-0.72")
        elif last_loss < 3.5:
            print(f"  ⚠️ Convergência mediana. Score esperado: 0.60-0.68")
        else:
            print(f"  ❌ Convergência fraca. Score esperado: < 0.60")

print("\n✅ CELL 5 COMPLETE")

In [ ]:
#@title 🔄 CELL 6: Manual Submit (checkpoint específico)
#@markdown Use esta célula para submeter um checkpoint manualmente.

CHECKPOINT_STEP = 400  #@param {type:"integer"}
SUBMIT_DESC = ""  #@param {type:"string"}

import glob as g

# Find checkpoint
ckpt_dir = f"{OUTPUT_DIR}/checkpoint-{CHECKPOINT_STEP}"
if not os.path.exists(ckpt_dir):
    # Try to find closest checkpoint
    ckpts = sorted(g.glob(f"{OUTPUT_DIR}/checkpoint-*"))
    print(f"Available checkpoints: {[os.path.basename(c) for c in ckpts]}")
    if ckpts:
        ckpt_dir = ckpts[-1]
        print(f"Using latest: {ckpt_dir}")
    else:
        print("❌ No checkpoints found!")
        ckpt_dir = None

if ckpt_dir and os.path.exists(ckpt_dir):
    zip_path = f"/tmp/kg1_submit/manual_step{CHECKPOINT_STEP}.zip"
    create_submission_zip(ckpt_dir, zip_path)
    
    if not SUBMIT_DESC:
        SUBMIT_DESC = f"{CFG['name']} manual-step-{CHECKPOINT_STEP} r{LORA_RANK}-a{LORA_ALPHA}"
    
    submit_to_kaggle(zip_path, SUBMIT_DESC)
    print("\n✅ Manual submit complete")

---

## v44 Production Recipe Summary

| Config | Value | Source |
|---|---|---|
| LoRA rank | 32 | Kaggle hard limit |
| LoRA alpha | 16 | v30/v41 PROVEN (0.68 LB) |
| LoRA dropout | 0.05 | v30/v41 + konbu17 |
| target_modules | all-linear | v30/v41 PROVEN |
| Learning rate | 1e-4 | konbu17 LB-validated |
| Max length | 4096 | konbu17 |
| Epochs | 2 | konbu17 |
| Grad accum | 8 | konbu17 |
| Warmup ratio | 0.05 | konbu17 |
| Scheduler | cosine | konbu17 |
| Seed | 123 | konbu17 exact |
| Samples | 2907 type-weighted | konbu17 |
| Liger-Kernel | YES (new) | QC3 research |
| Freeze router | YES | Aman Atar 168v |
| Ckpt averaging | YES | AIMO-2 free gain |
| Submit steps | [100,200,300,400] | Early feedback |
